# 🧪 시나리오 2-1: Round Robin 로드밸런싱

## 목적
APIM 백엔드 풀의 Round Robin 로드밸런싱이 정상 동작하는지 검증합니다:

1. **균등 분산** — 3개 백엔드로 요청이 고르게 분배되는지
2. **백엔드 추적** — `x-backend-url` 헤더로 실제 라우팅 대상 확인
3. **안정성** — 12회 연속 호출 시 모든 요청이 성공하는지

## 사전 조건
- Lab 1~2 완료 (APIM 배포 + Azure OpenAI 연결)
- 백엔드 풀 구성 완료 (3개 리전)
- outbound 정책에 `x-backend-url` 헤더 추가 완료
- 환경 변수 설정: `source scripts/env.sh`

In [ ]:
import os, time, json
from collections import Counter
import requests
from dotenv import load_dotenv

# scripts/env.sh에서 환경 변수 자동 로드
load_dotenv("../../scripts/env.sh")

APIM_URL = os.getenv("APIM_URL")
SUBSCRIPTION_KEY = os.getenv("APIM_SUBSCRIPTION_KEY")
DEPLOYMENT_NAME = os.getenv("DEPLOYMENT_NAME", "gpt-4.1-nano")
API_VERSION = "2025-04-01-preview"

assert APIM_URL, "❌ APIM_URL이 설정되지 않았습니다. scripts/env.sh를 확인하세요."
assert SUBSCRIPTION_KEY, "❌ APIM_SUBSCRIPTION_KEY가 설정되지 않았습니다. scripts/env.sh를 확인하세요."

BASE_URL = f"{APIM_URL}/openai/deployments/{DEPLOYMENT_NAME}/chat/completions"
HEADERS = {
    "Content-Type": "application/json",
    "Ocp-Apim-Subscription-Key": SUBSCRIPTION_KEY
}

print("✅ 환경 설정 완료")
print(f"   APIM URL:    {APIM_URL}")
print(f"   Deployment:  {DEPLOYMENT_NAME}")
print(f"   API Version: {API_VERSION}")

In [ ]:
print("▶ Round Robin 로드밸런싱 테스트 (12회 호출)\n")

N = 12
results = []
raw_responses = []

print(f"  {'#':>3}  {'Status':>6}  Backend")
print(f"  {'─' * 50}")

for i in range(N):
    body = {"messages": [{"role": "user", "content": f"Say number {i}"}], "temperature": 0, "max_tokens": 10}
    r = requests.post(
        BASE_URL,
        params={"api-version": API_VERSION},
        headers={**HEADERS, "x-client-request-id": f"rr-{i}-{int(time.time()*1000)}"},
        json=body,
        timeout=60
    )
    backend = r.headers.get("x-backend-url", "<missing>")
    results.append({"call": i + 1, "status": r.status_code, "backend": backend})

    # 전체 응답 저장 (헤더 + 바디)
    raw_responses.append({
        "call": i + 1,
        "status": r.status_code,
        "backend": backend,
        "headers": dict(r.headers),
        "body": r.json() if r.headers.get("content-type", "").startswith("application/json") else r.text
    })

    icon = "✅" if r.status_code == 200 else "❌"
    print(f"  {i+1:3d}  {icon} {r.status_code}  {backend}")
    time.sleep(0.3)

# JSON 로그 파일 저장
LOG_FILE = os.path.join(os.path.dirname(os.path.abspath("__file__")), "test-roundrobin-log.json")
with open(LOG_FILE, "w", encoding="utf-8") as f:
    json.dump(raw_responses, f, ensure_ascii=False, indent=2)

print(f"\n📝 전체 응답 로그 저장: {LOG_FILE}")

In [ ]:
print("═" * 55)
print(" Round Robin 로드밸런싱 결과")
print("═" * 55)

# 상태 코드 분포
status_count = Counter([x["status"] for x in results])
success_rate = status_count.get(200, 0) / len(results) * 100
print(f"\n  성공률: {status_count.get(200, 0)}/{len(results)} ({success_rate:.0f}%)")

# 백엔드 분포
print(f"\n  {'백엔드':<40} {'요청수':>5} {'비율':>6}")
print(f"  {'─' * 55}")
backend_count = Counter([x["backend"] for x in results])
for backend, count in backend_count.most_common():
    pct = count / len(results) * 100
    bar = "█" * int(pct / 5)
    print(f"  {backend:<40} {count:>5} {pct:>5.0f}% {bar}")

print(f"\n  사용된 백엔드: {len(backend_count)}개")

# 판정
print(f"\n{'─' * 55}")
checks = {
    "모든 요청 성공 (200)": all(x["status"] == 200 for x in results),
    "2개 이상 백엔드 분산": len(backend_count) >= 2,
    "3개 백엔드 균등 분산": len(backend_count) >= 3,
}
for check, passed in checks.items():
    icon = "✅" if passed else "❌"
    print(f"  {icon} {check}")

---
## ✅ 체크리스트

| # | 확인 항목 | 기대 결과 |
|---|-----------|----------|
| 1 | 12회 호출 모두 HTTP 200 | 100% 성공률 |
| 2 | `x-backend-url` 헤더 존재 | 모든 응답에 포함 |
| 3 | 2개 이상 백엔드로 분산 | 최소 2개 |
| 4 | 3개 백엔드 균등 분산 (33% ± 편차) | 이상적: 각 4회 |